# 13 MoE Routing and Experts

## Problem

MoE 为什么能把模型做大，而不让每个 token 都激活全部参数？router、expert、top-k routing、load balance 分别在解决什么问题？

## Dependencies

建议先完成 `08_ffn_and_swiglu.ipynb`，因为 MoE 常常可以先理解成“把一个 FFN 换成很多个专家 FFN”。


## Goals

- 理解 dense FFN 和 MoE FFN 的关键区别
- 理解 router 如何决定 token 该走哪些专家
- 理解 top-k routing、load balance、activated parameters 的含义
- 能用最小例子讲清楚 MoE 为什么能做大总容量

## Scope and Approach

这一节不会只说“MoE 就是很多专家”。我们会先从 dense FFN 的局限说起，再解释 router 到底在分配什么，最后用一个最小例子把 top-k routing 和专家聚合过程走一遍。


## 先从 dense FFN 的局限说起

在普通 Transformer 里，FFN 是 dense 的：

- 每个 token 都走同一套参数
- 不管这个 token 是代码、数学、自然语言还是符号串，都用同一个 FFN 处理

这样做当然简单，但当你想把模型总容量做大时，会遇到一个问题：

- 如果所有 token 每次都激活全部参数
- 计算成本也会跟着一起放大

MoE 的出发点就是：

**能不能把总参数做得很大，但每个 token 只激活其中一小部分？**


## MoE 的直觉

MoE 可以先粗糙理解成一个“分诊系统”：

- 准备很多个 expert
- 每个 expert 都像一套独立的 FFN
- 对每个 token，先由 router 判断更适合送给谁
- 只激活 top-k 个专家，而不是全部专家都跑一遍

所以 MoE 的关键不是“专家很多”这一点本身，而是：

- **总参数很多**
- **单次激活参数不一定很多**


In [ ]:
import numpy as np

np.random.seed(99)
np.set_printoptions(precision=3, suppress=True)

# 这里用 3 个 token 做示意，每个 token 4 维。
x = np.array([
    [0.2, 0.5, 0.1, 0.7],
    [0.9, 0.3, 0.4, 0.2],
    [0.1, 0.8, 0.6, 0.5],
])

num_experts = 3
hidden_dim = 4
ffn_dim = 6

def softmax(logits):
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / np.sum(exp, axis=-1, keepdims=True)


## router 到底在输出什么

router 不是在直接回答“这个 token 的最终表示是什么”，而是在回答：

**这个 token 更应该送给哪些 expert？**

它的输出通常是一组对 expert 的打分或概率。之后系统会：

- 选出 top-k 个 expert
- 只让这些 expert 真正参与计算

所以 router 更像一个调度器，而不是另一个内容生成模块。


In [ ]:
# router_w 把 token 表示映射成“这个 token 对各专家的偏好打分”。
router_w = np.random.randn(hidden_dim, num_experts) * 0.3
router_logits = x @ router_w
router_probs = softmax(router_logits)

print('router_logits =')
print(router_logits)
print('router_probs =')
print(router_probs)


## 每个 expert 本质上像什么

在最朴素的理解里，每个 expert 都可以被看成一套自己的 FFN 参数。也就是说：

- expert 1 有一套 FFN
- expert 2 有另一套 FFN
- expert 3 再有另一套 FFN

这样一来，不同 token 可以被送进不同参数路径里处理。


In [ ]:
experts = []
for _ in range(num_experts):
    w1 = np.random.randn(hidden_dim, ffn_dim) * 0.3
    w2 = np.random.randn(ffn_dim, hidden_dim) * 0.3
    experts.append((w1, w2))


## top-k routing 到底在做什么

top-k routing 的意思不是“每个 expert 都跑完再选最好的”，而是：

- 先用 router 给所有 expert 打分
- 只保留分数最高的 k 个 expert
- 只有这些 expert 真正参与前向计算

这样做的目的，是让单次计算成本别随着总 expert 数一起线性爆炸。


In [ ]:
outputs = []

for token_idx, token in enumerate(x):
    probs = router_probs[token_idx]

    # 这里只保留 top-2 个 expert。
    top_k = probs.argsort()[-2:][::-1]

    combined = np.zeros(hidden_dim)
    print(f'token {token_idx} -> experts {top_k}')

    for expert_id in top_k:
        w1, w2 = experts[expert_id]

        # 每个 expert 内部就像一个小 FFN。
        hidden = np.maximum(token @ w1, 0)
        expert_out = hidden @ w2

        # 用 router 概率作为聚合权重。
        combined += probs[expert_id] * expert_out

    outputs.append(combined)

outputs = np.stack(outputs)
print('moe outputs =')
print(outputs)


## 为什么还要关心 load balance

如果 router 总是把大多数 token 都送去同一个 expert，会发生什么？

- 某几个 expert 非常忙
- 另外一些 expert 几乎学不到东西
- 总容量虽然看起来很大，但真正被充分利用的部分很少

所以 MoE 里常常会额外关心 load balance，也就是：

**不要让路由过于偏科。**

这不是一个可选装饰，而是稀疏模型能不能训好的关键问题之一。


## total parameters 和 activated parameters 为什么要分开看

MoE 很容易在宣传里出现两种数字：

- total parameters：模型里一共放了多少参数
- activated parameters：单次前向时，真正被某个 token 用到多少参数

这两个数字必须分开看，因为 MoE 的核心卖点之一就是：

- **总容量可以很大**
- **单次激活不一定等于全部容量**

这也是 MoE 和普通 dense 模型最关键的差别之一。

## Common mistakes

- 把 MoE 想成“所有专家都跑一遍再平均”。关键恰恰在于只激活少数专家。
- 只关注总参数量，而忽略 activated parameters 才更接近一次前向真正用到的参数。
- 忽视路由平衡问题。如果所有 token 都挤向少数专家，MoE 会训练得很别扭。
- 以为 MoE 只是把模型做大。更准确地说，它是在“总容量”和“单次计算成本”之间重新做配置。

## Checkpoints

- 把 top-k 从 2 改成 1 或 3，想想容量和计算量会怎么变。
- 观察不同 token 被分到的 expert 是否一致。
- 用自己的话解释 router 和 expert 分别在做什么。
- 说明为什么 activated parameters 和 total parameters 要分开看。

## Summary

MoE 的核心价值，是把“总容量”和“单次激活成本”部分解耦。router 决定 token 走哪些专家，top-k routing 控制计算预算，load balance 保证专家不被严重偏科，activated parameters 则告诉你一次前向真正动用了多少模型容量。

## Next Module

下一节开始进入训练目标和训练流程。到那里，重点会从结构本身转向：这些结构到底是在什么样的数据和优化目标下被训练出来的。
